# Geração de Máscaras TIFF com Modificação e Cálculo de NDVI

Este notebook replica o comportamento de criação automática de máscaras TIFF usando os parâmetros de `args.yaml`.

**Características:**
- Carrega todas as orthoimages das 3 regiões com todos os seus canais
- Verifica quantas bandas cada região tem
- Calcula mapa de NDVI para cada região
- Identifica pixels com NDVI < 0
- Gera máscaras seguindo o mesmo processo do código principal
- Permite modificação das máscaras usando NDVI antes de salvá-las

## 1. Imports e Configuração

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import rasterio
from scipy.ndimage import binary_fill_holes
from skimage.morphology import convex_hull_image
from PIL import Image
import matplotlib.pyplot as plt

# Adicionar o diretório raiz ao path
# Calcular PROJECT_ROOT a partir do diretório atual
# Se estiver em exploration_notebooks, sobe 1 nível; se estiver na raiz, usa diretório atual
current_dir = Path(os.getcwd()).resolve()
if current_dir.name == 'exploration_notebooks':
    PROJECT_ROOT = current_dir.parent
elif 'exploration_notebooks' in str(current_dir):
    # Se estiver dentro de exploration_notebooks/subdir
    idx = str(current_dir).find('exploration_notebooks')
    PROJECT_ROOT = Path(str(current_dir)[:idx]).parent if idx > 0 else current_dir.parent.parent
else:
    # Tentar caminho absoluto como fallback
    PROJECT_ROOT = Path("/home/luizluz/Documentos/multi-task-fcn")

sys.path.insert(0, str(PROJECT_ROOT))

from src.io_operations import (
    read_yaml, 
    normalize_multi_region_args,
    fix_relative_paths,
    read_tiff,
    get_image_metadata,
    array2raster
)
from src.utils import check_folder
from utils import fast_imshow

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Diretório atual: {current_dir}")
print(f"sys.path[0]: {sys.path[0] if sys.path else 'N/A'}")

## 2. Carregar Parâmetros do args.yaml

In [ ]:
# Carregar args.yaml
args_path = PROJECT_ROOT / "args.yaml"
args = read_yaml(str(args_path))

# Normalizar paths relativos para absolutos
fix_relative_paths(args)

# Normalizar configuração multi-região
args = normalize_multi_region_args(args)

# Parâmetros para geração de máscara (padrões do código)
make_convex = True
fill_holes = True
save_preview = True
preview_max_size = 1024

# Configuração para cálculo de NDVI
# Banda 0 = Red, Banda 3 = NIR (ajuste se necessário)
RED_BAND_IDX = 0
NIR_BAND_IDX = 3

print(f"Número de regiões: {args.get('num_regions', len(args.get('ortho_images', [])))}")
print(f"Orthoimages:")
for i, ortho_path in enumerate(args.get('ortho_images', [])):
    print(f"  Região {i}: {ortho_path}")
print(f"\nConfiguração NDVI: Banda {RED_BAND_IDX} = Red, Banda {NIR_BAND_IDX} = NIR")

## 3. Carregar Todas as Orthoimages das 3 Regiões

In [ ]:
# Carregar todas as orthoimages com todos os canais
orthoimages = {}
orthoimages_metadata = {}

num_regions = args.get('num_regions', len(args.get('ortho_images', [])))

for region_idx in range(num_regions):
    ortho_path = args['ortho_images'][region_idx]
    
    # Garantir que o caminho seja absoluto
    if not Path(ortho_path).is_absolute():
        ortho_path = PROJECT_ROOT / ortho_path
        ortho_path = str(ortho_path.resolve())
    
    print(f"\n{'='*60}")
    print(f"Carregando região {region_idx}: {Path(ortho_path).name}")
    print(f"  Caminho completo: {ortho_path}")
    print(f"{'='*60}")
    
    # Verificar se o arquivo existe
    if not Path(ortho_path).exists():
        print(f"  ERRO: Arquivo não encontrado: {ortho_path}")
        continue
    
    # Carregar orthoimage com todos os canais
    ortho = read_tiff(ortho_path)
    
    # Obter metadados
    metadata = get_image_metadata(ortho_path)
    
    orthoimages[region_idx] = ortho
    orthoimages_metadata[region_idx] = metadata
    
    # print(f"  Shape: {ortho.shape}")
    # print(f"  Dtype: {ortho.dtype}")
    # print(f"  Min: {ortho.min()}, Max: {ortho.max()}")
    
    # if ortho.ndim == 3:
    #     num_bands = ortho.shape[0]
    #     print(f"  Número de bandas: {num_bands}")
    #     print(f"  Dimensões: {ortho.shape[1]} x {ortho.shape[2]} pixels")
        
    #     # Analisar cada banda
    #     print(f"\n  Análise por banda:")
    #     for i in range(num_bands):
    #         band = ortho[i]
    #         mean_val = band[band > 0].mean() if np.any(band > 0) else 0
    #         std_val = band[band > 0].std() if np.any(band > 0) else 0
    #         max_val = band.max()
    #         min_val = band[band > 0].min() if np.any(band > 0) else 0
    #         print(f"    Banda {i}: média={mean_val:.2f}, std={std_val:.2f}, min={min_val}, max={max_val}")
    # else:
    #     print(f"  Imagem monocromática")
    
    # print(f"  CRS: {metadata.get('crs')}")
    # print(f"  Transform: {metadata.get('transform')}")

## 4. Calcular NDVI para Cada Região

NDVI (Normalized Difference Vegetation Index) é calculado como:
$$NDVI = \frac{NIR - Red}{NIR + Red}$$

Onde:
- **NIR** = Near Infrared (infravermelho próximo)
- **Red** = Vermelho

Valores de NDVI variam de -1 a 1:
- **NDVI < 0**: Água, nuvens, neve, ou pixels inválidos
- **NDVI ≈ 0**: Solo exposto ou rochas
- **NDVI > 0**: Vegetação (quanto maior, mais saudável)

In [ ]:
def calculate_ndvi(red_band, nir_band):
    """
    Calcula NDVI a partir das bandas Red e NIR.
    
    Parameters
    ----------
    red_band : np.ndarray
        Banda vermelha (uint8)
    nir_band : np.ndarray
        Banda infravermelho próximo (uint8)
    
    Returns
    -------
    np.ndarray
        Mapa de NDVI (float32, valores entre -1 e 1)
    """
    # Converter para float32 para evitar overflow
    red = red_band.astype(np.float32)
    nir = nir_band.astype(np.float32)
    
    # Calcular NDVI: (NIR - Red) / (NIR + Red)
    # Evitar divisão por zero
    denominator = nir + red
    ndvi = np.where(denominator != 0, (nir - red) / denominator, 0.0)
    
    return ndvi

# Calcular NDVI para cada região
ndvi_maps = {}
ndvi_stats = {}

for region_idx in range(num_regions):
    ortho = orthoimages[region_idx]
    
    if ortho.ndim != 3:
        print(f"Região {region_idx}: Imagem não é multibanda, pulando cálculo de NDVI")
        continue
    
    num_bands = ortho.shape[0]
    
    # Verificar se as bandas necessárias existem
    if RED_BAND_IDX >= num_bands or NIR_BAND_IDX >= num_bands:
        print(f"Região {region_idx}: Bandas insuficientes para calcular NDVI")
        print(f"  Requeridas: Banda {RED_BAND_IDX} (Red) e Banda {NIR_BAND_IDX} (NIR)")
        print(f"  Disponíveis: {num_bands} bandas")
        continue
    
    print(f"\n{'='*60}")
    print(f"Calculando NDVI para região {region_idx}")
    print(f"{'='*60}")
    
    # Extrair bandas Red e NIR
    red_band = ortho[RED_BAND_IDX]
    nir_band = ortho[NIR_BAND_IDX]
    
    print(f"  Banda {RED_BAND_IDX} (Red): min={red_band.min()}, max={red_band.max()}, média={red_band[red_band>0].mean():.2f}")
    print(f"  Banda {NIR_BAND_IDX} (NIR): min={nir_band.min()}, max={nir_band.max()}, média={nir_band[nir_band>0].mean():.2f}")
    
    # Calcular NDVI
    ndvi = calculate_ndvi(red_band, nir_band)
    ndvi_maps[region_idx] = ndvi
    
    # Estatísticas
    valid_pixels = (red_band > 0) | (nir_band > 0)
    ndvi_valid = ndvi[valid_pixels]
    
    stats = {
        'min': float(ndvi_valid.min()),
        'max': float(ndvi_valid.max()),
        'mean': float(ndvi_valid.mean()),
        'std': float(ndvi_valid.std()),
        'negative_count': int(np.sum(ndvi < 0)),
        'negative_percent': float(100 * np.sum(ndvi < 0) / ndvi.size),
        'zero_count': int(np.sum(ndvi == 0)),
        'positive_count': int(np.sum(ndvi > 0)),
    }
    ndvi_stats[region_idx] = stats
    
    print(f"\n  Estatísticas NDVI:")
    print(f"    Min: {stats['min']:.4f}")
    print(f"    Max: {stats['max']:.4f}")
    print(f"    Média: {stats['mean']:.4f}")
    print(f"    Desvio padrão: {stats['std']:.4f}")
    print(f"\n  Distribuição:")
    print(f"    NDVI < 0: {stats['negative_count']:,} pixels ({stats['negative_percent']:.2f}%)")
    print(f"    NDVI = 0: {stats['zero_count']:,} pixels")
    print(f"    NDVI > 0: {stats['positive_count']:,} pixels")

In [ ]:
ortho.shape

In [ ]:
fast_imshow(np.moveaxis(ortho, 0, -1))

In [ ]:
fast_imshow(nir_band)

## 5. Visualizar Mapas de NDVI

In [ ]:
# Visualizar mapas de NDVI usando fast_imshow
fig, axes = plt.subplots(2, num_regions, figsize=(6*num_regions, 12))
if num_regions == 1:
    axes = axes.reshape(2, 1)

for region_idx in range(num_regions):
    if region_idx not in ndvi_maps:
        continue
    
    ndvi = ndvi_maps[region_idx]
    
    # Visualização 1: NDVI completo usando fast_imshow
    fast_imshow(ndvi, ax=axes[0, region_idx], cmap='RdYlGn', vmin=-1, vmax=1)
    axes[0, region_idx].set_title(f"Região {region_idx} - NDVI Completo\n{Path(args['ortho_images'][region_idx]).name}")
    axes[0, region_idx].axis('off')
    
    # Adicionar colorbar
    im1 = axes[0, region_idx].get_images()[0]
    plt.colorbar(im1, ax=axes[0, region_idx], label='NDVI')
    
    # Visualização 2: Pixels com NDVI < 0 destacados
    # Usar fast_imshow para NDVI base e depois adicionar overlay manualmente
    # Calcular escala usado pelo fast_imshow (padrão 1/16)
    scale = 1/16
    h, w = ndvi.shape[:2]
    new_h, new_w = int(h * scale), int(w * scale)
    
    # Redimensionar NDVI e máscara negativa
    ndvi_resized = cv2.resize(ndvi.astype(np.float32), (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    ndvi_negative_mask = ndvi < 0
    negative_mask_resized = cv2.resize(ndvi_negative_mask.astype(np.uint8), (new_w, new_h), interpolation=cv2.INTER_NEAREST)
    
    # Mostrar NDVI com colormap
    im2 = axes[1, region_idx].imshow(ndvi_resized, cmap='RdYlGn', vmin=-1, vmax=1, alpha=0.7)
    # Sobrepor máscara de pixels negativos em vermelho
    axes[1, region_idx].imshow(negative_mask_resized, cmap='Reds', alpha=0.3, vmin=0, vmax=1)
    axes[1, region_idx].set_title(f"Região {region_idx} - NDVI < 0 Destacado\n{ndvi_stats[region_idx]['negative_percent']:.2f}% dos pixels")
    axes[1, region_idx].axis('off')
    plt.colorbar(im2, ax=axes[1, region_idx], label='NDVI')

plt.tight_layout()
plt.show()

# Histograma de distribuição de NDVI
fig, axes = plt.subplots(1, num_regions, figsize=(6*num_regions, 4))
if num_regions == 1:
    axes = [axes]

for region_idx in range(num_regions):
    if region_idx not in ndvi_maps:
        continue
    
    ndvi = ndvi_maps[region_idx]
    valid_ndvi = ndvi[(ndvi != 0) & np.isfinite(ndvi)]
    
    axes[region_idx].hist(valid_ndvi, bins=100, alpha=0.7, edgecolor='black')
    axes[region_idx].axvline(x=0, color='r', linestyle='--', linewidth=2, label='NDVI = 0')
    axes[region_idx].set_xlabel('NDVI')
    axes[region_idx].set_ylabel('Frequência')
    axes[region_idx].set_title(f"Região {region_idx} - Distribuição de NDVI")
    axes[region_idx].legend()
    axes[region_idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Gerar Máscaras Iniciais

In [ ]:
# Gerar máscaras iniciais seguindo o mesmo processo do código
masks = {}

for region_idx in range(num_regions):
    ortho = orthoimages[region_idx]
    
    print(f"\n{'='*60}")
    print(f"Gerando máscara inicial para região {region_idx}...")
    print(f"{'='*60}")
    
    # Handle different shapes: (bands, height, width) or (height, width)
    if ortho.ndim == 3:
        # Multiband image: pixel is non-empty if ANY channel is non-zero
        mask = np.any(ortho != 0, axis=0)
        print(f"  Multiband image com shape {ortho.shape}")
    else:
        # Single band image
        mask = ortho != 0
        print(f"  Single band image com shape {ortho.shape}")
    
    # Count initial valid pixels
    initial_valid = np.sum(mask)
    total_pixels = mask.size
    print(f"  Pixels válidos iniciais: {initial_valid:,} / {total_pixels:,} ({100*initial_valid/total_pixels:.2f}%)")
    
    # Fill holes in the mask
    if fill_holes:
        mask = binary_fill_holes(mask)
        print(f"  Após fill_holes: {np.sum(mask):,} pixels válidos")
    
    # Make the mask convex
    if make_convex:
        mask = convex_hull_image(mask)
        print(f"  Após convex_hull: {np.sum(mask):,} pixels válidos")
    
    # Convert to uint8 (0 and 1)
    mask = mask.astype(np.uint8)
    
    masks[region_idx] = mask
    
    print(f"  Máscara final shape: {mask.shape}, dtype: {mask.dtype}")

## 7. Visualização das Máscaras Antes da Modificação

In [ ]:
# Visualizar máscaras geradas
fig, axes = plt.subplots(1, num_regions, figsize=(6*num_regions, 6))
if num_regions == 1:
    axes = [axes]

for region_idx in range(num_regions):
    mask = masks[region_idx]
    
    # Redimensionar para visualização se muito grande
    max_size = 2000
    if mask.shape[0] > max_size or mask.shape[1] > max_size:
        scale = min(max_size / mask.shape[0], max_size / mask.shape[1])
        new_h = int(mask.shape[0] * scale)
        new_w = int(mask.shape[1] * scale)
        mask_vis = Image.fromarray((mask * 255).astype(np.uint8), mode='L')
        mask_vis = mask_vis.resize((new_w, new_h), Image.Resampling.NEAREST)
        mask_vis = np.array(mask_vis) / 255.0
    else:
        mask_vis = mask
    
    axes[region_idx].imshow(mask_vis, cmap='gray')
    axes[region_idx].set_title(f"Região {region_idx} - Máscara Inicial\n{Path(args['ortho_images'][region_idx]).name}")
    axes[region_idx].axis('off')

plt.tight_layout()
plt.show()

## 8. Modificação das Máscaras Usando NDVI

**ESPAÇO PARA SUAS MODIFICAÇÕES**

Aqui você pode modificar as máscaras antes de salvá-las. As máscaras estão disponíveis no dicionário `masks` e os mapas de NDVI em `ndvi_maps`.

**Exemplos de uso do NDVI:**
- Remover pixels com NDVI < 0 da máscara (água, nuvens, pixels inválidos)
- Filtrar apenas áreas com vegetação (NDVI > threshold)
- Combinar máscara original com filtro NDVI

In [ ]:
# ============================================
# ESPAÇO PARA SUAS MODIFICAÇÕES
# ============================================
# Modifique as máscaras aqui antes de salvá-las
# As máscaras estão em masks[0], masks[1], masks[2]
# Os mapas de NDVI estão em ndvi_maps[0], ndvi_maps[1], ndvi_maps[2]

# Exemplo 1: Remover pixels com NDVI < 0 da máscara
# Descomente e ajuste conforme necessário:
"""
for region_idx in range(num_regions):
    if region_idx in ndvi_maps:
        mask = masks[region_idx].copy()
        ndvi = ndvi_maps[region_idx]
        
        # Remover pixels com NDVI < 0
        mask[ndvi < 0] = 0
        
        print(f"Região {region_idx}: Removidos {np.sum(ndvi < 0):,} pixels com NDVI < 0")
        masks[region_idx] = mask
"""

# Exemplo 2: Filtrar apenas áreas com vegetação (NDVI > threshold)
# Descomente e ajuste conforme necessário:
"""
NDVI_THRESHOLD = 0.1  # Ajuste o threshold conforme necessário

for region_idx in range(num_regions):
    if region_idx in ndvi_maps:
        mask = masks[region_idx].copy()
        ndvi = ndvi_maps[region_idx]
        
        # Manter apenas pixels com NDVI > threshold
        mask[ndvi <= NDVI_THRESHOLD] = 0
        
        print(f"Região {region_idx}: Mantidos apenas pixels com NDVI > {NDVI_THRESHOLD}")
        masks[region_idx] = mask
"""

# Exemplo 3: Operações morfológicas
# Descomente e ajuste conforme necessário:
"""
from scipy import ndimage
from skimage.morphology import remove_small_objects

for region_idx in range(num_regions):
    mask = masks[region_idx].copy()
    
    # Exemplo: aplicar erosão
    # mask = ndimage.binary_erosion(mask, structure=np.ones((5,5))).astype(np.uint8)
    
    # Exemplo: aplicar dilatação
    # mask = ndimage.binary_dilation(mask, structure=np.ones((5,5))).astype(np.uint8)
    
    # Exemplo: aplicar opening
    # mask = ndimage.binary_opening(mask, structure=np.ones((3,3))).astype(np.uint8)
    
    # Exemplo: aplicar closing
    # mask = ndimage.binary_closing(mask, structure=np.ones((3,3))).astype(np.uint8)
    
    # Exemplo: remover pequenos objetos
    # mask = remove_small_objects(mask.astype(bool), min_size=1000).astype(np.uint8)
    
    masks[region_idx] = mask
"""

# Exemplo 4: Combinar filtro NDVI com operações morfológicas
# Descomente e ajuste conforme necessário:
"""
from scipy import ndimage

NDVI_THRESHOLD = 0.0  # Remover apenas pixels com NDVI < 0

for region_idx in range(num_regions):
    if region_idx in ndvi_maps:
        mask = masks[region_idx].copy()
        ndvi = ndvi_maps[region_idx]
        
        # Remover pixels com NDVI < threshold
        mask[ndvi < NDVI_THRESHOLD] = 0
        
        # Aplicar operações morfológicas para limpar a máscara
        # mask = ndimage.binary_opening(mask, structure=np.ones((3,3))).astype(np.uint8)
        # mask = ndimage.binary_closing(mask, structure=np.ones((5,5))).astype(np.uint8)
        
        masks[region_idx] = mask
        print(f"Região {region_idx}: Máscara modificada com filtro NDVI")
"""

print("="*60)
print("Máscaras prontas para modificação.")
print("Descomente e ajuste os exemplos acima ou adicione seu próprio código.")
print("="*60)

## 9. Visualização das Máscaras Após Modificação (Opcional)

In [ ]:
# Visualizar máscaras após modificação (opcional)
# Descomente se quiser visualizar após suas modificações usando fast_imshow

# fig, axes = plt.subplots(1, num_regions, figsize=(6*num_regions, 6))
# if num_regions == 1:
#     axes = [axes]
# 
# for region_idx in range(num_regions):
#     mask = masks[region_idx]
#     
#     # Converter para uint8 para visualização
#     mask_vis = (mask * 255).astype(np.uint8)
#     
#     # Usar fast_imshow para visualização eficiente
#     fast_imshow(mask_vis, ax=axes[region_idx], cmap='gray', vmin=0, vmax=255)
#     axes[region_idx].set_title(f"Região {region_idx} - Máscara Modificada\n{Path(args['ortho_images'][region_idx]).name}")
#     axes[region_idx].axis('off')
# 
# plt.tight_layout()
# plt.show()

## 10. Salvar Máscaras como TIFF

In [ ]:
# Determinar pasta de saída para máscaras geradas
data_path = args.get('data_path', '.')
masks_folder = Path(data_path) / "generated_masks"
check_folder(str(masks_folder))

print(f"Pasta de saída: {masks_folder}")

# Salvar máscaras
saved_mask_paths = {}

for region_idx in range(num_regions):
    mask = masks[region_idx]
    metadata = orthoimages_metadata[region_idx]
    ortho_path = args['ortho_images'][region_idx]
    
    # Gerar nome do arquivo baseado no nome da orthoimage
    from os.path import basename, splitext
    ortho_name = splitext(basename(ortho_path))[0]
    output_path = masks_folder / f"{ortho_name}_mask.tif"
    
    print(f"\n{'='*60}")
    print(f"Salvando máscara região {region_idx}...")
    print(f"{'='*60}")
    print(f"  Arquivo: {output_path}")
    print(f"  Shape: {mask.shape}")
    print(f"  Dtype: {mask.dtype}")
    print(f"  Pixels válidos: {np.sum(mask):,} / {mask.size:,} ({100*np.sum(mask)/mask.size:.2f}%)")
    
    # Salvar TIFF
    array2raster(
        str(output_path), 
        mask, 
        metadata, 
        dtype='uint8'
    )
    
    saved_mask_paths[region_idx] = str(output_path)
    print(f"  ✓ Máscara salva com sucesso!")
    
    # Salvar preview PNG (opcional)
    if save_preview:
        png_path = output_path.with_suffix('.png').with_name(f"{output_path.stem}_preview.png")
        
        # Converter máscara para 0-255 para visualização
        mask_vis = (mask * 255).astype(np.uint8)
        
        # Criar imagem PIL
        img = Image.fromarray(mask_vis, mode='L')
        
        # Calcular dimensões de redimensionamento mantendo aspect ratio
        width, height = img.size
        if width > height:
            if width > preview_max_size:
                new_width = preview_max_size
                new_height = int(height * preview_max_size / width)
            else:
                new_width, new_height = width, height
        else:
            if height > preview_max_size:
                new_height = preview_max_size
                new_width = int(width * preview_max_size / height)
            else:
                new_width, new_height = width, height
        
        # Redimensionar se necessário
        if (new_width, new_height) != (width, height):
            img = img.resize((new_width, new_height), Image.Resampling.NEAREST)
        
        # Salvar PNG
        img.save(str(png_path), optimize=True)
        print(f"  ✓ Preview PNG salvo: {png_path.name} (tamanho: {new_width}x{new_height})")

print("\n" + "="*60)
print("TODAS AS MÁSCARAS FORAM SALVAS COM SUCESSO!")
print("="*60)
for region_idx, path in saved_mask_paths.items():
    print(f"Região {region_idx}: {path}")

## 11. Salvar Mapas de NDVI (Opcional)

Se desejar salvar os mapas de NDVI calculados, descomente o código abaixo.

In [ ]:
# Salvar mapas de NDVI (opcional)
# Descomente se quiser salvar os mapas de NDVI

# ndvi_folder = Path(data_path) / "ndvi_maps"
# check_folder(str(ndvi_folder))
# 
# print(f"\nSalvando mapas de NDVI em: {ndvi_folder}")
# 
# for region_idx in range(num_regions):
#     if region_idx not in ndvi_maps:
#         continue
#     
#     ndvi = ndvi_maps[region_idx]
#     metadata = orthoimages_metadata[region_idx]
#     ortho_path = args['ortho_images'][region_idx]
#     
#     # Gerar nome do arquivo
#     from os.path import basename, splitext
#     ortho_name = splitext(basename(ortho_path))[0]
#     output_path = ndvi_folder / f"{ortho_name}_ndvi.tif"
#     
#     # Converter NDVI para int16 para salvar (valores entre -10000 e 10000, representando -1.0 a 1.0)
#     # Ou salvar como float32
#     ndvi_to_save = (ndvi * 10000).astype(np.int16)  # Multiplicar por 10000 para preservar precisão
#     
#     # Criar metadados modificados para float32
#     metadata_ndvi = metadata.copy()
#     metadata_ndvi['dtype'] = 'int16'
#     
#     array2raster(
#         str(output_path),
#         ndvi_to_save,
#         metadata_ndvi,
#         dtype='int16'
#     )
#     
#     print(f"  Região {region_idx}: {output_path}")
# 
# print("Mapas de NDVI salvos!")